In [1]:
'''
Getting the ABDU model to work in notebook

EPSG: 5070
'''
import duckdb #version 1.1.3
import geopandas as gpd #version 0.14.1
import time
from shapely import wkt
# import pandas as pd
# import pyarrow as pa
import rasterio
from rasterio import mask
from shapely.geometry import shape
from threading import Thread, current_thread

con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")
con.install_extension("json")
con.load_extension("json")
print(duckdb.__version__) #previously 0.9.2

1.1.3


# Parameters

In [2]:
start_time = time.time()
time.ctime(start_time)

'Wed Sep 10 12:26:58 2025'

In [3]:
aoi = '28001'
nwiurl = r"azure://abdu/nwi/**/*.parquet"
wetattrfld = 'ATTRIBUTE'
waterurl = True
politicalbndry = r'azure://abdu/uscounties.parquet'
pid_fld = 'FIPS'
hucsurl = r'azure://abdu/huc/**/*.parquet'
hucidfld = 'huc12'
crossWalk_json = 'https://giscog.blob.core.windows.net/abdu/aoiWetland.json'
nrgy_csv = r'azure://abdu/kcal.csv'
fowlDemo = r'azure://abdu/WaterfowlDemographic.parquet'
protLands = r"azure://abdu/padus/**/*.parquet",
urbanMask = "https://giscog.blob.core.windows.net/privatecogs/NLCD2016_cog.tif",
demand = r'azure://abdu/Demand9SpeciesMerged.parquet'
in_crs = dict() # default is 'ESPG:5070'. All will be reprojectd to crs of hucsurl. Provide param:crs pairs if source crs differs from default.


In [4]:
# Parameters
pid_fld = "dgr_blk"
nwiurl = "D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_weth2o.parquet"
wetattrfld = "CLASS_NAME"
waterurl = True
hucsurl = "D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_wsheds_wkb.parquet"
hucidfld = "WATERSHED_CODE"
crossWalk_json = "https://giscog.blob.core.windows.net/abdu/caWetlands.json"
nrgy_csv = "azure://abdu/ehjv_kcal.csv"
protLands = "D:\\ABDUBounds\\ABDU_Canada_Data\\abdu_ca_protLands.parquet"
urbanMask = "D:\\ABDUBounds\\abdu_ca_urban.parquet"
demand = "azure://abdu/Demand9Species_Merged.parquet"
in_crs = {"D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_weth2o.parquet": "ESRI:102008", "D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_wsheds_wkb.parquet": "ESRI:102008"}
aoi = [[2185875.6240893, 3236133.5592382113], [2151850.858574032, 3340311.6055248305], [2221060.1946121375, 3363319.133820226], [2256179.192264037, 3259505.0832735235], [2185875.6240893, 3236133.5592382113]]


In [5]:
params = (nwiurl, politicalbndry, hucsurl, fowlDemo, protLands, urbanMask, demand, crossWalk_json, nrgy_csv)
if any([i in '_'.join(params) for i in ('azure','giscog')]):
    con.sql("SET azure_transport_option_type = 'curl'")
    con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')

In [6]:
param_crs = {p:'EPSG:5070' for p in params[:-2]}
if len(in_crs)>0:
    for k, v in in_crs.items():
            param_crs[k] = v
param_crs 

{'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_weth2o.parquet': 'ESRI:102008',
 'azure://abdu/uscounties.parquet': 'EPSG:5070',
 'D:\\ABDUBounds\\ABDU_Canada_Data\\parquet\\abdu_cn_east_wsheds_wkb.parquet': 'ESRI:102008',
 'azure://abdu/WaterfowlDemographic.parquet': 'EPSG:5070',
 'D:\\ABDUBounds\\ABDU_Canada_Data\\abdu_ca_protLands.parquet': 'EPSG:5070',
 'D:\\ABDUBounds\\abdu_ca_urban.parquet': 'EPSG:5070',
 'azure://abdu/Demand9Species_Merged.parquet': 'EPSG:5070'}

In [7]:
### transform/reproject geometry when necessary
def tfrm_str(geom='geometry', in_crs='EPSG:5070', out_crs=param_crs[hucsurl]):
    """
    format str for sql query to reproject geometry of duckdb table
    """
    if not in_crs==out_crs:
        s = f"ST_Transform({geom}, '{in_crs}', '{out_crs}')"
    else:
        s = geom
    return s
### Should add check for linear unit for hectare calculations

In [8]:
'''
SELECT fips geometry based on inaoifile to use as aoi for calculation.  All hucs should have center in fips.
'''
if isinstance(aoi, str):
    con.sql("""
        CREATE OR REPLACE TABLE selectedcounty AS
        SELECT {2}, geometry FROM read_parquet('{1}')
        WHERE {2} = '{0}'
    """.format(aoi, politicalbndry, pid_fld))
else:
    con.sql(f"CREATE OR REPLACE TABLE selectedcounty ({pid_fld} VARCHAR, geometry GEOMETRY)")
    xy = f"LINESTRING{tuple([' '.join(map(str,i)) for i in aoi])}".replace("'","")
    aoi = tfrm_str(f"ST_MakePolygon('{xy}')")
    con.sql(f"INSERT INTO selectedcounty (geometry) VALUES ({aoi})")
    xy = con.sql(f"""SELECT {tfrm_str('ST_Centroid(geometry)', param_crs[hucsurl], 'EPSG:4326')} AS center,
        ST_X(center) AS x, ST_Y(center) AS y,        
        FROM selectedcounty
        """).to_df()[['x','y']].values[0] ## for some reason X and Y are reversed = try 'always_xy' parameter
    aoi = f"n{int(xy[0])}_e{int(xy[1])}".replace('e-','w').replace('n-','s')
    # con.sql(f"INSERT INTO selectedcounty ({pid_fld}) VALUES ('{aoi}')")
    con.sql(f"UPDATE selectedcounty SET {pid_fld} = '{aoi}'")
    print(aoi)

n49_w65


In [9]:
'''
Demand

Read in demand based on centroid within selected county
'''
con.sql("""
CREATE OR REPLACE TABLE demand AS 
SELECT * EXCLUDE geometry, {1} AS geometry
FROM read_parquet('{0}') dmnd
JOIN selectedcounty ON ST_Within({2}, selectedcounty.geometry)
""".format(demand, tfrm_str('dmnd.geometry', param_crs[demand]), tfrm_str('ST_Centroid(dmnd.geometry)', param_crs[demand]))
)
# con.sql(f"""
# CREATE OR REPLACE TABLE demand AS
# SELECT {hucidfld}, CODE, species, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, ST_Area(geometry)*0.0001 AS ha, geometry
# FROM demand
# """)
        # FROM (SELECT * FROM demand
# WHERE species='All')
# """)
# spp_list = sorted([i[0] for i in con.sql(f"SELECT distinct(species) FROM read_parquet('{demand}')").fetchall()])
# spp_list
con.sql("SELECT * FROM demand")

┌──────────┬─────────┬────────────┬─────────┬────────┬────────┬───────────┬───────────┬──────────────┬────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [10]:
spp_list = sorted([i[0] for i in con.sql(f"SELECT distinct(species) FROM read_parquet('{demand}')").fetchall()])
spp_list

['ABDU', 'AGWT', 'AMWI', 'All', 'BWTE', 'GADW', 'MALL', 'NOPI', 'NSHO', 'WODU']

In [11]:
if '**' in hucsurl:
    '''
    Read in hucs partitioned to huc2/huc4 level that have center with the aoi.  Don't clip hucs
    '''
    sql = f"""
        CREATE OR REPLACE TABLE huc12 AS
        SELECT LEFT(huc12,2) AS huc2,LEFT(huc12,4) AS huc4, huc12, areaacres, huc.geometry
        FROM (SELECT huc12, areaacres, ST_GeomFromWKB(read_parquet.geometry) AS geometry FROM read_parquet('{hucsurl}', hive_partitioning=true)
        WHERE CAST(LEFT(huc12,2) AS INTEGER)<=12) AS huc
        JOIN selectedcounty ON 
        ST_Within(ST_Centroid(huc.geometry), selectedcounty.geometry)
    """
    con.sql(sql)
    hucs = con.sql("select huc4 from huc12 GROUP BY huc4").df().values.tolist()
    hucs = sorted([item for items in hucs for item in items])
    hucidfld = 'huc4'
    wet_flds = [wetattrfld, 'huc2', hucidfld, 'huc12']
    con.execute("""
        CREATE OR REPLACE TABLE my_wetlands (
            {0} VARCHAR,
            huc2 VARCHAR,
            {1} VARCHAR,
            huc12 VARCHAR,
            geometry VARCHAR,
        )
    """.format(wetattrfld, hucidfld))
else:
    sql = f"""
        CREATE OR REPLACE TABLE huc12 AS
        SELECT huc.* EXCLUDE geometry, ST_Intersection(huc.geometry, selectedcounty.geometry) AS geometry, selectedcounty.{pid_fld}
        FROM read_parquet('{hucsurl}') AS huc
        JOIN selectedcounty ON 
        ST_Intersects(huc.geometry, selectedcounty.geometry)
    """
    con.sql(sql)
    hucs = con.sql(f"SELECT {hucidfld} from huc12").df().values.tolist()
    hucs = sorted([item for items in hucs for item in items])
    wet_flds = [wetattrfld, hucidfld]
    con.execute("""
        CREATE OR REPLACE TABLE my_wetlands (
            {0} VARCHAR,
            {1} VARCHAR,
            geometry VARCHAR,
        )
    """.format(wetattrfld, hucidfld))
len(hucs), hucs

(20,
 ['NS01585',
  'NS01586',
  'NS01589',
  'NS01590',
  'NS01591',
  'NS01603',
  'NS01604',
  'NS02069',
  'NS02070',
  'NS02071',
  'NS02075',
  'NS02076',
  'NS02077',
  'NS02078',
  'NS02079',
  'NS02081',
  'NS02082',
  'NS02083',
  'NS02084',
  'NS02085'])

In [12]:
'''
Import Waterfowl demographic data to assign a fips to a specific code (breeding [4b] and non-breeding [4d])
'''
code = con.sql(f"""
SELECT code FROM read_parquet('{fowlDemo}')
JOIN selectedcounty ON 
ST_Within(ST_Centroid(selectedcounty.geometry), {tfrm_str('read_parquet.geometry')})
""").df().values.tolist()
if len(code)>0:
    code = code[0][0]
else:
    code = '4b' # Use latitude to determine?
if code is None:
    code = '4b'
print(code)

4b


# Wetland energy calculation


In [13]:
def write_from_thread(con):
    local_con = con.cursor()
    huc = str(current_thread().name)
    if any([i in nwiurl for i in ('azure','giscog')]):
        local_con.sql('''CREATE OR REPLACE SECRET secret0 (TYPE AZURE, ACCOUNT_NAME 'giscog')''')
        local_con.sql("SET azure_transport_option_type = 'curl'")
    if '**' in nwiurl:
        sql = '''
        SELECT {2}, ST_AsWKB(ST_Intersection(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(huc12.geometry)) AS geometry
        FROM (SELECT {3}, geometry FROM read_parquet('{1}',hive_partitioning=true) 
        WHERE {4} = '{0}' AND NOT ({3} LIKE 'R%UB%' OR {3} LIKE 'R%SB%' OR {3} LIKE 'R%RB%')) AS wetlnd
        JOIN huc12 ON 
        ST_Intersects(ST_GeomFromWKB(wetlnd.geometry), ST_GeomFromWKB(huc12.geometry)))
        '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld)
        rows = local_con.sql(sql).fetchall()
    else:
        if waterurl:
            sql = """
            SELECT {2}, ST_AsText(ST_Intersection({5}, huc12.geometry)) AS geometry
            FROM read_parquet('{1}') wet, huc12
            WHERE huc12.{4} = '{0}'
            AND ST_Intersects({5}, huc12.geometry)
            AND NOT ST_IsEMpty(ST_Intersection({5}, huc12.geometry))
            """.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
            rows = local_con.sql(sql).fetchall()
            if isinstance(waterurl, str):
                sql = '''
                WITH wet AS
                    ({5}),
                h2o AS
                    (SELECT StandardClass, ST_Intersection(h2o.geometry, huc12.geometry) AS geometry FROM read_parquet('{1}') h2o, huc12
                    WHERE huc12.{4} = '{0}'
                    AND ST_Intersects(h2o.geometry, huc12.geometry))
                SELECT StandardClass, {4}, ST_AsText(ST_Difference(h2o.geometry, ST_GeomFromText(wet.geometry))) AS geometry
                FROM h2o, wet
                WHERE ST_Intersects(h2o.geometry, ST_GeomFromText(wet.geometry))
                AND NOT ST_IsEMpty(ST_Difference(h2o.geometry, ST_GeomFromText(wet.geometry)))
                '''.format(huc, waterurl, ', '.join(wet_flds), wetattrfld, hucidfld, sql)
                rows.extend(local_con.sql(sql).fetchall())                 
        else:
            sql = '''
                SELECT {2}, ST_AsText(ST_Intersection({5}, huc12.geometry)) AS geometry
                FROM read_parquet('{1}') wet, huc12
                WHERE huc12.{4} = '{0}'
                AND wet.{3} != 'Open Water'
                AND ST_Intersects({5}, huc12.geometry)
                AND NOT ST_IsEMpty(ST_Intersection({5}, huc12.geometry))
                '''.format(huc, nwiurl, ', '.join(wet_flds), wetattrfld, hucidfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
            rows = local_con.sql(sql).fetchall()
    sql = f'''INSERT INTO my_wetlands 
    VALUES ({'?, '*len(wet_flds)}?)
    '''
    if len(rows)>0:
        result = local_con.executemany(sql, rows).fetchall()

In [14]:
threads = []
print(len(hucs),hucs)
for i in range(len(hucs)):
    huc = hucs[i]
    threads.append(Thread(target = write_from_thread,
                            args = (con,),
                            name = huc))
print(len(threads))

20 ['NS01585', 'NS01586', 'NS01589', 'NS01590', 'NS01591', 'NS01603', 'NS01604', 'NS02069', 'NS02070', 'NS02071', 'NS02075', 'NS02076', 'NS02077', 'NS02078', 'NS02079', 'NS02081', 'NS02082', 'NS02083', 'NS02084', 'NS02085']
20


In [15]:
%%time
# Kick off all threads in parallel
for thread in threads:
    thread.start()

# Ensure all threads complete before printing final results
for thread in threads:
    thread.join()

# con.sql("""
#     CREATE OR REPLACE TABLE wetlands AS 
#     SELECT * FROM my_wetlands 
# """)

CPU times: total: 5min 35s
Wall time: 18.6 s


In [16]:
'''
Import wetland crossclass data and assign classes to the nwi table
'''
con.sql(f"""CREATE OR REPLACE TABLE crossnwi AS
        (UNPIVOT (FROM (SELECT * FROM read_json_auto('{crossWalk_json}', maximum_object_size=100000000))) ON COLUMNS(*))""")
con.sql("""CREATE OR REPLACE TABLE crossnwi AS
        SELECT name, UNNEST(value) AS value FROM crossnwi""")
con.sql(f"""CREATE OR REPLACE TABLE wetlands AS
        SELECT name, {wet_flds[-1]}, ST_GeomFromText(geometry) AS geometry FROM my_wetlands
        LEFT JOIN crossnwi ON my_wetlands.{wetattrfld} LIKE crossnwi.value
        """)
con.sql(f"""
        CREATE OR REPLACE TABLE wetlands AS
        SELECT replace(wetlands.name, '_', '') AS name, {wet_flds[-1]}, ST_Area(geometry)*0.0001 AS ha, kcal, kcal*ha AS avalNrgy, st_buffer(geometry,0) AS geometry FROM wetlands
        LEFT JOIN read_csv_auto('{nrgy_csv}') ON replace(wetlands.name, '_', '') = read_csv_auto.habitatType
        WHERE wetlands.name IS NOT NULL
        """)
print(con.sql('SELECT count(name) FROM wetlands'))

┌───────────────┐
│ count("name") │
│     int64     │
├───────────────┤
│          1766 │
└───────────────┘



# Protected Lands

In [17]:
'''
Read in PADUS
'''
if '**' in protLands:
    sql = """
    CREATE OR REPLACE TABLE protected AS 
    SELECT CATEGORY, huc12, huc2, huc4, ST_Intersection(ST_GeomFromWKB(huc12.geometry), ST_GeomFromWKB(prot.geometry)) as geometry
    FROM (SELECT CATEGORY, geometry FROM read_parquet('{1}', hive_partitioning=true)
    WHERE CATEGORY IN ('Fee', 'Easements', 'Other') AND huc4 IN {0}) AS prot
    JOIN huc12 ON 
    ST_Intersects(ST_GeomFromWKB(huc12.geometry), ST_GeomFromWKB(prot.geometry))
    """.format(tuple(hucs), protLands)
else:
    sql = """
    CREATE OR REPLACE TABLE protected AS
    SELECT {1}, ST_Intersection(huc12.geometry, {2}) as geometry
    FROM read_parquet('{0}') AS prot
    JOIN huc12 ON 
    ST_Intersects(huc12.geometry, {2})
    """.format(protLands, hucidfld, tfrm_str('prot.geometry'))
con.sql(sql)

In [18]:
con.sql("""
CREATE OR REPLACE TABLE protwetlands AS
SELECT name, wetlands.{0}, kcal, ST_Intersection(protected.geometry, wetlands.geometry) as geometry
FROM (SELECT ST_Union_Agg(geometry) as geometry from protected) as protected
JOIN wetlands ON 
ST_Intersects(wetlands.geometry, protected.geometry)
""".format(hucidfld))

In [19]:
con.sql("""
CREATE OR REPLACE TABLE protwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS ProtHabHa, kcal, kcal*ProtHabHa AS protNrgy FROM protwetlands
""".format(hucidfld))

# Urban wetlands

In [20]:
'''
Read in NLCD clipped to hucs
'''
if 'nlcd' in urbanMask.lower() and urbanMask.endswith('.tif'):
    # Need huc12 geometry
    df = con.sql('SELECT ST_AsText(ST_geomfromwkb(geometry)) as geometry from huc12').df()
    df['geometry'] = df['geometry'].apply(wkt.loads)
    df = gpd.GeoDataFrame(df, geometry='geometry', crs=5070)
    with rasterio.open(urbanMask) as src:
        # Clip the raster to the geometry of the shapefile
        clipped_data, transform = mask.mask(src, df.geometry, crop=True)
    del df
    clipped_data[clipped_data>23]=0
    clipped_data[clipped_data<21]=0
    clipped_data[clipped_data==21]=1
    clipped_data[clipped_data==22]=1
    clipped_data[clipped_data==23]=1
    shapes = rasterio.features.shapes(clipped_data[0], transform=transform, mask=clipped_data[0] == 1)
    # Create a GeoDataFrame from the vector polygons
    gdf_vector = gpd.GeoDataFrame({'geometry': [shape(geom) for geom, value in shapes]})
    gdf_vector['geometry'] = gdf_vector.to_wkb().geometry
    sql = "CREATE OR REPLACE TABLE urban AS SELECT * EXCLUDE geometry, ST_GeomFromWKB(geometry) AS geometry FROM gdf_vector"
else:
    sql = """
    CREATE OR REPLACE TABLE urban AS
    SELECT ST_Intersection(huc12.geometry, {1}) as geometry
    FROM read_parquet('{0}') AS urb_mask
    JOIN huc12 ON ST_Intersects(huc12.geometry, {1})
    """.format(urbanMask, tfrm_str('urb_mask.geometry', param_crs[urbanMask]))
con.sql(sql)

In [21]:
con.sql("""
CREATE OR REPLACE TABLE urban AS 
SELECT {0}, ST_Intersection(huc12.geometry, urban.geometry) as geometry
FROM (SELECT geometry FROM urban) as urban
JOIN huc12 ON ST_Intersects(huc12.geometry, urban.geometry)
""".format(hucidfld))

In [22]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT name, wetlands.{0}, kcal, ST_Intersection(urban.geometry, wetlands.geometry) as geometry
FROM (SELECT geometry from urban) as urban
JOIN wetlands ON ST_Intersects(wetlands.geometry, urban.geometry)
""".format(hucidfld))

In [23]:
con.sql("""
CREATE OR REPLACE TABLE urbanwetlands AS
SELECT DISTINCT geometry, name, {0}, ST_Area(geometry)*0.0001 AS ha, kcal, kcal*ha AS urbanNrgy FROM urbanwetlands
""".format(hucidfld))

In [24]:
con.sql("""
CREATE OR REPLACE TABLE urban AS
SELECT {0}, ST_Area(geometry)*0.0001 AS urbanHa, geometry FROM urban
""".format(hucidfld))

In [25]:
con.sql("""
CREATE OR REPLACE TABLE unavailable AS
SELECT {0}, ST_Area(geometry)*0.0001 AS unavailHa, ST_Union_Agg(geometry) as geometry FROM
(
SELECT {0}, geometry FROM urban
UNION ALL
SELECT {0}, geometry from protected
)
group by {0}, geometry
""".format(hucidfld))

In [26]:
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
#################################
'''
#################################
End of data import
Starting model process
#################################
'''
#### Prepping energy - Join energy to nwi.  Need to create the spatial kcal table first. What's the best way to do this?
# parquet is the best to read in but it's not easily editable.  Rest service would be ok but again, not great because
# reading those is difficult.  I wonder if

'\n#################################\nEnd of data import\nStarting model process\n#################################\n'

In [27]:
con.sql("SHOW TABLES")

┌────────────────┐
│      name      │
│    varchar     │
├────────────────┤
│ crossnwi       │
│ demand         │
│ huc12          │
│ my_wetlands    │
│ protected      │
│ protwetlands   │
│ selectedcounty │
│ unavailable    │
│ urban          │
│ urbanwetlands  │
│ wetlands       │
├────────────────┤
│    11 rows     │
└────────────────┘

# Demand calculation

In [28]:
'''
Demand

######
Need to proportion demand based on available energy.  Available energy is spatially explicit but demand is at the fips level.  
We need to calculate total energy and demand at the huc12 scale.
To proportion demand we need to calclulate total energy by fips then calculate how much energy is in each huc12. A proportion
can then be calculated by dividing total energy within a fips by (huc12,fips) group.  Demand at the huc12 level is multiplied
by that energy proportion.
######
'''

'\nDemand\n\n######\nNeed to proportion demand based on available energy.  Available energy is spatially explicit but demand is at the fips level.  \nWe need to calculate total energy and demand at the huc12 scale.\nTo proportion demand we need to calclulate total energy by fips then calculate how much energy is in each huc12. A proportion\ncan then be calculated by dividing total energy within a fips by (huc12,fips) group.  Demand at the huc12 level is multiplied\nby that energy proportion.\n######\n'

In [29]:
i = con.sql("SELECT count(*) FROM demand").fetchone()[0]
i

3

In [30]:
if i > 0:
        if waterurl:
                sql = """
                SELECT {1}, {2} AS geometry
                FROM read_parquet('{0}') wet, demand
                WHERE species = 'All'
                AND ST_Intersects({2}, demand.geometry)
                AND NOT ST_IsEMpty(ST_Intersection({2}, demand.geometry))
                """.format(nwiurl, wetattrfld, tfrm_str('wet.geometry', param_crs[nwiurl]))            
        else:
                sql = """
                SELECT {1}, {2} AS geometry
                FROM read_parquet('{0}') wet, demand
                WHERE species = 'All'
                AND ST_Intersects({2}, demand.geometry)
                AND NOT ST_IsEMpty(ST_Intersection({2}, demand.geometry))
                AND wet.{1} != 'Open Water'
                """.format(nwiurl, wetattrfld, tfrm_str('wet.geometry', param_crs[nwiurl]))
        con.execute(
                f"""
                CREATE OR REPLACE TABLE demandNrgy AS
                WITH dmndwetlands AS ({sql})
                SELECT * EXCLUDE geometry, ST_Intersection(demand.geometry, dmndwetlands.geometry) AS geometry FROM demand, dmndwetlands
                WHERE species = 'All' AND CODE = '{code.upper()}'
                """
        )
        print(con.sql("SELECT count(*) FROM demandNrgy"))
        con.sql("SELECT * EXCLUDE geometry FROM demandNrgy LIMIT 10")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1742 │
└──────────────┘



# Energy Calculation

In [31]:
'''Get sum of energy within huc12'''
energysum = con.sql('select sum(avalNrgy) from wetlands').fetchone()[0]
print(energysum)

258433505.7496193


In [32]:
if i>0:
        con.sql(f"""CREATE OR REPLACE TABLE demandNrgy AS
                SELECT * FROM demandNrgy
                LEFT JOIN crossnwi ON demandNrgy.{wetattrfld} LIKE crossnwi.value
                """)
        con.sql(f"""
                CREATE OR REPLACE TABLE demandNrgy AS
                SELECT * EXCLUDE name, replace(demandNrgy.name, '_', '') AS name, ST_Area(geometry)*0.0001 AS ha, kcal*ha AS avalNrgy
                FROM demandNrgy
                LEFT JOIN read_csv_auto('{nrgy_csv}') ON replace(demandNrgy.name, '_', '') = read_csv_auto.habitatType
                WHERE demandNrgy.name IS NOT NULL
                """)
        energysumfromdemand = con.sql('SELECT sum(avalNrgy) FROM demandNrgy').fetchone()
        print(energysumfromdemand)

(258369843.6273177,)


In [33]:
if i > 0:
    con.sql(f"""CREATE OR REPLACE TABLE hucdemandenergy AS 
        (SELECT demandNrgy.{pid_fld}, name, {hucidfld}, kcal, 
        ST_Intersection(huc12.geometry, demandNrgy.geometry) as geometry FROM huc12, demandNrgy
        )""")

In [34]:
'''
######
Calculate available energy (avalNrgy) of wetlands by calculating area in Hectares (HA) and multiplying by kcal.
Select only distinct rows.
Create new table habitatenergy
######
'''
if i > 0:
        con.sql(f"""CREATE OR REPLACE TABLE hucdemandenergy AS
                (SELECT DISTINCT {pid_fld}, name, {hucidfld}, ST_Area(geometry)*0.0001 AS ha, ha*kcal AS avalNrgy 
                FROM hucdemandenergy)
                """)
        con.sql(f"""CREATE OR REPLACE TABLE hucdemandenergy AS
                SELECT {pid_fld},{hucidfld}, sum(ha) AS totalhab_ha, sum(avalNrgy) as avalNrgy FROM hucdemandenergy GROUP BY ({pid_fld},{hucidfld})
                """)
        energysumfromdemand = con.sql("select sum(avalNrgy) from hucdemandenergy").fetchone()[0]
else:
        energysumfromdemand = i

In [35]:
'''Total of availenergy'''
'''Get sum of energy within huc12'''
print('Wetland energy: {:,.2f}'.format(energysum))
print('Demand energy: {:,.2f}'.format(energysumfromdemand))
dif = energysumfromdemand - energysum
print('Difference: {:,.0%}'.format(abs(dif/((energysumfromdemand + energysum)/2))))


Wetland energy: 258,433,505.75
Demand energy: 257,862,474.29
Difference: 0%


In [36]:
if energysumfromdemand > 0:
        con.sql(f"""CREATE OR REPLACE TABLE rdydemand AS 
                WITH hucsNrgy AS (
                        SELECT {pid_fld}, {hucidfld}, sum(avalNrgy) AS avalNrgy
                        FROM hucdemandenergy
                        GROUP BY ({hucidfld}, {pid_fld})
                ), dmndTotal AS (
                        SELECT {pid_fld}, sum(avalNrgy) AS totalNrgy
                        FROM demandNrgy
                        GROUP BY {pid_fld}
                )
                SELECT *, avalNrgy/totalNrgy AS pct
                FROM hucsNrgy
                JOIN dmndTotal ON hucsNrgy.{pid_fld}=dmndTotal.{pid_fld}
                """)
        con.sql(f"""CREATE OR REPLACE TABLE rdydemand AS 
                SELECT {hucidfld}, rdydemand.{pid_fld}, species, LTADUD, LTADemand, LTAPopObj, x80DUD, X80Demand, X80PopObj, avalNrgy, pct
                FROM rdydemand
                JOIN demand ON rdydemand.{pid_fld}=demand.{pid_fld}
                """)


In [37]:
#con.sql("""describe rdydemand""")
if energysumfromdemand > 0:
    con.sql(f"""CREATE OR REPLACE TABLE hucdemand AS (SELECT {hucidfld}, species,
        sum(pct * LTADUD) AS LTADUD,
        sum(pct * LTADemand) AS LTADemand,
        sum(pct * LTAPopObj) AS LTAPopObj,
        sum(pct * x80DUD) AS x80DUD,
        sum(pct * X80Demand) AS X80Demand,
        sum(pct * X80PopObj) AS X80PopObj,
        FROM rdydemand
        GROUP BY {hucidfld}, species)
        """)
else:
    con.sql(f"""CREATE OR REPLACE TABLE hucdemand AS
            SELECT {hucidfld} FROM huc12
            """)
    cols = ['species', 'LTADUD', 'LTADemand', 'LTAPopObj', 'x80DUD', 'X80Demand', 'X80PopObj']
    for col in cols:
        if col == cols[0]:
            col = f"{col} VARCHAR DEFAULT 'All'"
        else:
            col = f"{col} INTEGER DEFAULT 0"
        con.sql(f"""ALTER TABLE hucdemand
                ADD {col}
                """)

    # con.sql(f"""SELECT {hucidfld}, LTADemand FROM hucdemand""")

In [38]:
con.sql('''CREATE OR REPLACE TABLE spp_demand AS
(SELECT * FROM
(PIVOT hucdemand
    on species
    USING {0}))
'''.format(', '.join([f"sum({i})" for i in con.sql("DESCRIBE hucdemand").df().column_name.to_list()[2:]])))

In [39]:
# ', '.join([i.replace("sum(","").replace(")","") for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:] if 'All' not in i])
for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:]:
    con.sql(f"""ALTER TABLE spp_demand RENAME COLUMN '{i}' TO '{i.replace('sum(','').replace(')','')}'""")
con.sql("SELECT * FROM spp_demand")

┌────────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬──────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬────────────────────────┐
│ WATERSHED_CODE │    ABDU_LTADUD     │   ABDU_LTADemand   │    ABDU_LTAPopObj     │     ABDU_x80DUD     │   ABDU_X80Demand   │    ABDU_X80PopObj    │     All_LTADUD      │   All_LTADemand    │     All_LTAPopObj     │     All_x80DUD      │   All_X80Demand    │     All_X80PopObj     │     MALL_LTADUD     │   MALL_LTADemand   │    MALL_LTAPopObj     │     MALL_x80DUD     │   MALL_X80Demand   │     MALL_X80PopObj     │
│    varchar     │       double       │       double       │        double         │       double        │       double       │        double        │       dou

In [ ]:
cols = con.sql('describe spp_demand').df()['column_name'].tolist()
print(cols)
for i in spp_list:
    if i.lower()=='all':
        continue
    for j in [f"{i}_{k}" for k in con.sql("DESCRIBE demand").df()['column_name'].tolist() if k.lower().startswith('lta') or k.lower().startswith('x80')]:
        if j.lower() not in map(str.lower, cols):
            con.sql(f"ALTER TABLE spp_demand ADD COLUMN {j} DOUBLE")
print(con.sql('describe spp_demand').df()['column_name'].tolist())

['WATERSHED_CODE', 'ABDU_LTADUD', 'ABDU_LTADemand', 'ABDU_LTAPopObj', 'ABDU_x80DUD', 'ABDU_X80Demand', 'ABDU_X80PopObj', 'All_LTADUD', 'All_LTADemand', 'All_LTAPopObj', 'All_x80DUD', 'All_X80Demand', 'All_X80PopObj', 'MALL_LTADUD', 'MALL_LTADemand', 'MALL_LTAPopObj', 'MALL_x80DUD', 'MALL_X80Demand', 'MALL_X80PopObj']
['WATERSHED_CODE', 'ABDU_LTADUD', 'ABDU_LTADemand', 'ABDU_LTAPopObj', 'ABDU_x80DUD', 'ABDU_X80Demand', 'ABDU_X80PopObj', 'All_LTADUD', 'All_LTADemand', 'All_LTAPopObj', 'All_x80DUD', 'All_X80Demand', 'All_X80PopObj', 'MALL_LTADUD', 'MALL_LTADemand', 'MALL_LTAPopObj', 'MALL_x80DUD', 'MALL_X80Demand', 'MALL_X80PopObj', 'AGWT_LTADUD', 'AGWT_X80DUD', 'AGWT_LTAPopObj', 'AGWT_X80PopObj', 'AGWT_LTADemand', 'AGWT_X80Demand', 'AMWI_LTADUD', 'AMWI_X80DUD', 'AMWI_LTAPopObj', 'AMWI_X80PopObj', 'AMWI_LTADemand', 'AMWI_X80Demand', 'BWTE_LTADUD', 'BWTE_X80DUD', 'BWTE_LTAPopObj', 'BWTE_X80PopObj', 'BWTE_LTADemand', 'BWTE_X80Demand', 'GADW_LTADUD', 'GADW_X80DUD', 'GADW_LTAPopObj', 'GADW_X8

# Summarize at huc level

In [41]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT huc12.{hucidfld}, huc12.{pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, huc12.geometry
FROM huc12
LEFT JOIN hucdemand on hucdemand.{hucidfld} = huc12.{hucidfld}
WHERE species = 'All'
ORDER by huc12.{hucidfld}
""")

In [42]:
# Specified selection in a cell or two below.  Many don't need geometry at this later point.  Joining is by huc12 so selecting
# only the required columns makes the join go much faster.
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, sum(avalNrgy) as tothabitat_kcal, athuclevel.geometry
FROM athuclevel
LEFT JOIN wetlands on wetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [43]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, 
sum(urbanHa) as urbanHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN urban on urban.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [44]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa,
sum(protNrgy) as protected_kcal,
sum(ProtHabHa) as protectedhabitat_ha,
sum(protNrgy) as protected_kcal,
athuclevel.geometry
FROM athuclevel
LEFT JOIN protwetlands on protwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, athuclevel.geometry
ORDER by athuclevel.{hucidfld}
""")

In [45]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal,
sum(urbanNrgy) as urbanNrgy,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, urbanNrgy FROM urbanwetlands) as urbanwetlands on urbanwetlands.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal, athuclevel.geometry
""")

In [46]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal,urbanNrgy,
sum(unavailHa) as unavailHa,
athuclevel.geometry
FROM athuclevel
LEFT JOIN (SELECT {hucidfld}, unavailHa FROM unavailable) as unavailable on unavailable.{hucidfld} = athuclevel.{hucidfld}
GROUP BY athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal, urbanNrgy, athuclevel.geometry
""")

In [47]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld}, {pid_fld}, LTADUD, LTADemand, LTAPopObj, X80DUD, X80Demand, X80PopObj, tothabitat_kcal, urbanHa, protectedhabitat_ha, protected_kcal,urbanNrgy, unavailHa,
{','.join([i for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:] if not 'All' in i])}, athuclevel.geometry FROM athuclevel
LEFT JOIN spp_demand on spp_demand.{hucidfld} = athuclevel.{hucidfld}
ORDER by athuclevel.{hucidfld}
""")

In [48]:
con.sql(f"""CREATE OR REPLACE TABLE athuclevel AS
SELECT athuclevel.{hucidfld},
ST_Area(geometry)*0.0001 wshed_ha,
{pid_fld}, 
COALESCE(LTADUD, 0) dud_lta,
COALESCE(LTADemand,0) demand_lta_kcal, 
COALESCE(LTAPopObj,0) popobj_lta, 
COALESCE(X80DUD,0) dud_80th, 
COALESCE(X80Demand,0) demand_80th_kcal, 
COALESCE(X80PopObj,0) popobj_80th,
{','.join([f'COALESCE({i}, 0) {i.lower()}' for i in con.sql("DESCRIBE spp_demand").df().column_name.to_list()[1:] if not 'All' in i])}, 
COALESCE(tothabitat_kcal,0) tothabitat_kcal,
COALESCE(protected_kcal,0) protected_kcal,
COALESCE(protectedhabitat_ha,0) protectedhabitat_ha,
COALESCE(urbanHa,0) urbanHa, 
COALESCE(sum(urbanNrgy),0) urbanNrgy,
COALESCE(sum(unavailHa),0) unavailha,
COALESCE(tothabitat_kcal - demand_lta_kcal,0) surpdef_lta_kcal,
COALESCE(tothabitat_kcal - demand_80th_kcal,0) surpdef_80th_kcal,
athuclevel.geometry
FROM athuclevel
GROUP BY *
ORDER BY athuclevel.{hucidfld}
""")

In [49]:
con.sql('''CREATE OR REPLACE TABLE athuclevel AS 
SELECT *,
CASE WHEN 
demand_lta_kcal - protected_kcal > 0
THEN
demand_lta_kcal - protected_kcal
ELSE 0
END
AS nrgprot_lta_kcal,
CASE WHEN
demand_80th_kcal - protected_kcal > 0 
THEN
demand_80th_kcal - protected_kcal
ELSE 0
END
AS nrgprot_80th_kcal
FROM athuclevel
''')

# Calculate pct kcal, ha by habitat type for weighted mean in post

In [ ]:
'''
Calculate mean energy per ha weighted by wetland type
'''
con.sql(f'''
        CREATE OR REPLACE TABLE wtmean AS 
        SELECT huctotal.{hucidfld}, name, tothab_ha, (habname_ha/tothab_ha)*100 as pct_ha, (avalNrgname/avalNrgtot)*100 as pct_kcal FROM
        ((SELECT {hucidfld}, sum(ha) as tothab_ha, sum(avalNrgy) as avalNrgtot from wetlands group by {hucidfld}) huctotal
        join
        (SELECT {hucidfld}, name, kcal, sum(ha) as habname_ha, sum(avalNrgy) as avalNrgname from wetlands group by {hucidfld}, name, kcal) hucnametotal
        on hucnametotal.{hucidfld} = huctotal.{hucidfld})
''')
con.sql(f'''CREATE OR REPLACE TABLE habpivot AS
(select * FROM
(pivot wtmean
    on name
    USING sum(pct_kcal) AS pct_kcal, sum(pct_ha) AS pct_ha))
''')

In [57]:
cols = con.sql('describe habpivot').df()['column_name'].tolist()
# for i in ('DeepwaterFresh', 'FreshMarsh', 'SaltMarshNonDominant', 'FreshShallowOpenWater', 'FreshwaterWoody', 'ManagedFreshMarsh', 'ManagedFreshShallowOpenWater', 'ManagedFreshwaterAquaticBed'):
for i in con.sql("SELECT distinct(name) FROM crossnwi").fetchall():
    j = f"{i[0].replace('_','')}_pct_kcal"
    if j not in cols:
        con.sql(f"ALTER TABLE habpivot ADD COLUMN {j} DOUBLE")
    j = j.replace("_pct_kcal","_pct_ha")
    if j not in cols:
        con.sql(f"ALTER TABLE habpivot ADD COLUMN {j} DOUBLE")
con.sql(f"""SELECT * FROM habpivot ORDER BY {hucidfld}""")

┌────────────────┬──────────────────────┬─────────────────────────┬───────────────────────┬──────────────────────┬─────────────────────┬──────────────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬───────────────────────────────┬─────────────────────────────┬────────────────────────────────┬──────────────────────────────┐
│ WATERSHED_CODE │      tothab_ha       │ DeepwaterFresh_pct_kcal │ DeepwaterFresh_pct_ha │ FreshMarsh_pct_kcal  │  FreshMarsh_pct_ha  │ FreshwaterWoody_pct_kcal │ FreshwaterWoody_pct_ha │ MudflatSalt_pct_kcal │ MudflatSalt_pct_ha  │ SaltMarshNonDominant_pct_kcal │ SaltMarshNonDominant_pct_ha │ FreshShallowOpenWater_pct_kcal │ FreshShallowOpenWater_pct_ha │
│    varchar     │        double        │         double          │        double         │        double        │       double        │          double          │         double         │        double        │       double        │            double             │           doub

In [58]:
con.sql(f"""CREATE OR REPLACE TABLE habbyhuc AS 
        SELECT {hucidfld}, tothab_ha,
        {', '.join([f"sum(coalesce({i},0)) AS {i}" for i in con.sql("DESCRIBE habpivot").df().column_name.to_list()[2:]])}
        FROM habpivot
        GROUP BY {hucidfld}, tothab_ha
        ORDER BY {hucidfld}
""")
con.sql(f"""SELECT * FROM habbyhuc""")

┌────────────────┬──────────────────────┬─────────────────────────┬───────────────────────┬──────────────────────┬─────────────────────┬──────────────────────────┬────────────────────────┬──────────────────────┬─────────────────────┬───────────────────────────────┬─────────────────────────────┬────────────────────────────────┬──────────────────────────────┐
│ WATERSHED_CODE │      tothab_ha       │ DeepwaterFresh_pct_kcal │ DeepwaterFresh_pct_ha │ FreshMarsh_pct_kcal  │  FreshMarsh_pct_ha  │ FreshwaterWoody_pct_kcal │ FreshwaterWoody_pct_ha │ MudflatSalt_pct_kcal │ MudflatSalt_pct_ha  │ SaltMarshNonDominant_pct_kcal │ SaltMarshNonDominant_pct_ha │ FreshShallowOpenWater_pct_kcal │ FreshShallowOpenWater_pct_ha │
│    varchar     │        double        │         double          │        double         │        double        │       double        │          double          │         double         │        double        │       double        │            double             │           doub

In [60]:
con.sql(f'''CREATE OR REPLACE TABLE athuclevel AS
SELECT * 
FROM athuclevel
LEFT JOIN habbyhuc on athuclevel.{hucidfld}=habbyhuc.{hucidfld}
order by athuclevel.{hucidfld}
''')

In [ ]:
## Calculate protect/restore goals in post

# Save output to file

In [ ]:
'''
Protected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of
wetland energy

Calculations:
    Energy supply
        Total habitat energy within huc - THabNrg
        Total habitat hectares within huc - THabHA

    Energy demand
        LTA and X80 DUD by huc - TLTADUD anc X80DUD
        LTA and X80 Demand by huc - TLTADemand and X80Demand
        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj
        
    Protected lands
        Total protected hectares by huc - ProtHA

    Protected habitat hectares and energy
        Total protected hectares - ProtHabHA
        Total protected energy - ProtHabNrg

    Weighted mean and calculations based off of it
        Weighted mean kcal/ha with weight being Total habitat energy
        Energy Protection needed - NrgProtRq
        Restoration HA based off of weighted mean - RstorHA
        Protection HA based off weighted mean - RstorProtHA  

'''
#################################
#################################
#################################


'\nProtected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of\nwetland energy\n\nCalculations:\n    Energy supply\n        Total habitat energy within huc - THabNrg\n        Total habitat hectares within huc - THabHA\n\n    Energy demand\n        LTA and X80 DUD by huc - TLTADUD anc X80DUD\n        LTA and X80 Demand by huc - TLTADemand and X80Demand\n        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj\n        \n    Protected lands\n        Total protected hectares by huc - ProtHA\n\n    Protected habitat hectares and energy\n        Total protected hectares - ProtHabHA\n        Total protected energy - ProtHabNrg\n\n    Weighted mean and calculations based off of it\n        Weighted mean kcal/ha with weight being Total habitat energy\n        Energy Protection needed - NrgProtRq\n        Restoration HA based off of weighted mean - RstorHA\n        Protection HA based off weighted mean - RstorProtHA 

In [ ]:
con.sql("DESCRIBE athuclevel").df().column_name.to_list()
con.sql("SELECT * EXCLUDE geometry FROM athuclevel")

┌────────────────┬────────────────────┬─────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬──────────────────────┬─────────────────────┬────────────────────┬───────────────────────┬─────────────────────┬────────────────────┬────────────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────

In [ ]:
# aoi = f"{aoi}_recalc"
aoi

'n49_w65'

In [ ]:
con.sql("""COPY (SELECT * FROM athuclevel) TO './output/{0}.parquet' (FORMAT PARQUET)""".format(aoi))
print('Done in {0:.1f} seconds'.format(time.time() - start_time))

Done in 198.6 seconds
